# Data Cleaning Notebook
This notebook handles:
- Missing values
- Standardizing date formats
- Deduplicating records

Load your dummy CSVs in `/opt/airflow/data`.

In [2]:
import pandas as pd
import numpy as np

DATA_DIR = "data"

files = {
    "routes": "dummy_routes.csv",
    "shelter_corridor": "dummy_shelter_corridor.csv",
    "realisasi_bus": "dummy_realisasi_bus.csv",
    "transaksi_bus": "dummy_transaksi_bus.csv",
    "transaksi_halte": "dummy_transaksi_halte.csv",
}

dfs = {name: pd.read_csv(f"{DATA_DIR}/{file}") for name, file in files.items()}
dfs

{'routes':    route_code                             route_name
 0           1                          Blok M - Kota
 1           2                    Pulo Gadung - Monas
 2           3                      Kalideres - Monas
 3           4               Pulo Gadung - Galunggung
 4           5                 Kampung Melayu - Ancol
 5           6                   Ragunan - Galunggung
 6           7      Kampung Rambutan - Kampung Melayu
 7           8               Lebak Bulus - Pasar Baru
 8           9                   Pinang Ranti - Pluit
 9          10                    Tanjung Priok - PGC
 10         11           Pulo Gebang - Kampung Melayu
 11         12                  Tanjung Priok - Pluit
 12         13              Puri Beta - Tegal Mampang
 13         14  Jakarta International Stadium - Senen
 14        B21              Blok M - Bundaran Senayan
 15        C12                       Cawang - Harmoni
 16        D11                     Depok - Dukuh Atas
 17        F11    

## 1. Handling Missing Values

In [3]:
cleaned = {}
for name, df in dfs.items():
    print(f"\n=== {name} ===")
    print("Missing values BEFORE:")
    print(df.isna().sum())

    df_clean = df.fillna({
        col: "" if df[col].dtype == 'object' else 0 
        for col in df.columns
    })

    print("Missing values AFTER:")
    print(df_clean.isna().sum())
    cleaned[name] = df_clean


=== routes ===
Missing values BEFORE:
route_code    0
route_name    0
dtype: int64
Missing values AFTER:
route_code    0
route_name    0
dtype: int64

=== shelter_corridor ===
Missing values BEFORE:
shelter_name_var    0
corridor_code       0
corridor_name       0
dtype: int64
Missing values AFTER:
shelter_name_var    0
corridor_code       0
corridor_name       0
dtype: int64

=== realisasi_bus ===
Missing values BEFORE:
shelter_name_var    0
corridor_code       0
corridor_name       0
dtype: int64
Missing values AFTER:
shelter_name_var    0
corridor_code       0
corridor_name       0
dtype: int64

=== transaksi_bus ===
Missing values BEFORE:
uuid                  0
waktu_transaksi       0
armada_id_var         0
no_body_var           0
card_number_var       0
card_type_var         0
balance_before_int    0
fare_int              0
balance_after_int     0
transcode_txt         0
gate_in_boo           0
p_latitude_flo        0
p_longitude_flo       0
status_var            0
free_service

## 2. Standardizing Date Columns

In [4]:
date_columns = ["waktu_transaksi", "insert_on_dtm", "tanggal_realisasi"]

for name, df in cleaned.items():
    for col in date_columns:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    cleaned[name] = df
cleaned

{'routes':    route_code                             route_name
 0           1                          Blok M - Kota
 1           2                    Pulo Gadung - Monas
 2           3                      Kalideres - Monas
 3           4               Pulo Gadung - Galunggung
 4           5                 Kampung Melayu - Ancol
 5           6                   Ragunan - Galunggung
 6           7      Kampung Rambutan - Kampung Melayu
 7           8               Lebak Bulus - Pasar Baru
 8           9                   Pinang Ranti - Pluit
 9          10                    Tanjung Priok - PGC
 10         11           Pulo Gebang - Kampung Melayu
 11         12                  Tanjung Priok - Pluit
 12         13              Puri Beta - Tegal Mampang
 13         14  Jakarta International Stadium - Senen
 14        B21              Blok M - Bundaran Senayan
 15        C12                       Cawang - Harmoni
 16        D11                     Depok - Dukuh Atas
 17        F11    

## 3. Deduplication

In [5]:
deduped = {}
for name, df in cleaned.items():
    before = len(df)
    df2 = df.drop_duplicates(keep="last")
    after = len(df2)
    print(f"{name}: removed {before-after} duplicates")
    deduped[name] = df2
deduped

routes: removed 0 duplicates
shelter_corridor: removed 0 duplicates
realisasi_bus: removed 0 duplicates
transaksi_bus: removed 0 duplicates
transaksi_halte: removed 0 duplicates


{'routes':    route_code                             route_name
 0           1                          Blok M - Kota
 1           2                    Pulo Gadung - Monas
 2           3                      Kalideres - Monas
 3           4               Pulo Gadung - Galunggung
 4           5                 Kampung Melayu - Ancol
 5           6                   Ragunan - Galunggung
 6           7      Kampung Rambutan - Kampung Melayu
 7           8               Lebak Bulus - Pasar Baru
 8           9                   Pinang Ranti - Pluit
 9          10                    Tanjung Priok - PGC
 10         11           Pulo Gebang - Kampung Melayu
 11         12                  Tanjung Priok - Pluit
 12         13              Puri Beta - Tegal Mampang
 13         14  Jakarta International Stadium - Senen
 14        B21              Blok M - Bundaran Senayan
 15        C12                       Cawang - Harmoni
 16        D11                     Depok - Dukuh Atas
 17        F11    